# Using the AniSOM implementation

## Installation Steps

download or clone the repository from [here](https://github.com/phrenico/maco_commondriver/tree/submission) note that it is the "submission" branch:

```bash
$git clone https://github.com/phrenico/maco_commondriver/tree/submission
```


create a new environment and install python

```bash
$conda create -n anisom_env python

```


then activate the environment
```bash
$conda activate anisom_env
```


Now we need to install the package. We need to navigate to the directory where the package is located and install it using the following command:
```bash
$cd maco_commondriver
$pip install -e .
```


It can take a while to install the package. Once the package is installed, we can use it in our code.

## Imports

In [3]:
import sys
path_to_cdriver = "/home/zsiga/Projects/Codes/maco_commondriver/"  # CHANGE IT TO YOUR PATH
sys.path.append(path_to_cdriver)
from cdriver.network.anisom import AniSOM
from cdriver.preprocessing.tde import time_delay_embedding
from cdriver.evaluate.evalz import comp_ccorr, get_maxes
from cdriver.datagen.logmap import gen_logmapdata

import numpy as np
import torch

## Parameters

In [4]:
# Parameters
# Define parameters and layers for deep model
d_embed = 3
d_grid = 2
d_space = d_embed
sizes = [40, 20]

logmapgen_params = dict(N=1,  # number of realizations
                        n=10_000,  # Length of time series
                        rint=(3.8, 4.),  # interval to chose from the value of r parameter
                        A0=np.array([[0, 0, 0],
                                     [1, 0, 0],
                                     [1, 0, 0]]),  # basic connection structure
                        A=np.array([[1., 0., 0.],
                                    [0.3, 1., 0.],
                                    [0.4, 0., 1.]]))

nof_epochs= 1  # number of epochs

## Generate Data

In [5]:
# Data Generation
(dataset, params) = gen_logmapdata(logmapgen_params)

print("parameters: ", params)
print("dataset shape: ", dataset[0].shape)

data = dataset[0]  # first realization from the dataset (in this case we had only one realization)

# Hidden driver extraction and Time Delay Embedding
z = data[:-(d_embed - 1), 0]
X = time_delay_embedding(data[:, 1], dimension=d_embed)
Y = time_delay_embedding(data[:, 2], dimension=d_embed)

Generating dataset: 100%|██████████| 1/1 [00:00<00:00,  3.26it/s]

parameters:  ({'r': array([3.9097627 , 3.94303787, 3.92055268]), 'A': array([[1. , 0. , 0. ],
       [0.3, 1. , 0. ],
       [0.4, 0. , 1. ]]), 'x0': array([0.54488318, 0.4236548 , 0.64589411])},)
dataset shape:  (10000, 3)


## ANISOM

In [6]:
# Apply AniSOM
ani = AniSOM(space_dim=d_space, grid_dim=d_grid, sizes=sizes)

ani.fit(torch.Tensor(X), torch.Tensor(Y), epochs=nof_epochs, disable_tqdm=True)
pred = ani.predict(torch.Tensor(X))

In [7]:
display(pred.shape)

print("Maximum predicted values:", pred.max(axis=0))

torch.Size([9998, 2])

Maximum predicted values: torch.return_types.max(
values=tensor([39, 19]),
indices=tensor([26, 15]))


In [9]:
# Evaluation
tau, c = comp_ccorr(pred[:, 1], z)

maxtau, maxc = get_maxes(tau, c)


print("Max correlation: ", maxc)
print("Max correlation was at tau: ", maxtau)

Max correlation:  0.8347629763283959
Max correlation was at tau:  -1
